## Process the customers Data
- Ingest the data into the data lakehouse - bronze_customers
- Perform data quality checks and transform the data as required - silver_customer_clean
- Apply changes to the customers data - silver_customers

In [0]:
-- 1. Ingest the data into the data lakehouse - bronze_customers

create or refresh streaming table bronze_customers
comment "Raw customers data ingested from the source system operational data"
tblproperties ('quality' = 'bronze')
as
select *,
        _metadata.file_path as input_file_path,
        current_timestamp as ingestion_timestamp
from cloud_files(
    '/Volumes/circuitbox/landing/operational_data/customers/',
    'json',
    map('cloudfiles.inferColumnTypes', 'true')
);


In [0]:
CREATE OR REFRESH STREAMING TABLE silver_customer_clean(
    CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
    CONSTRAINT valid_customer_name EXPECT (customer_name IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT valid_telephone EXPECT (LENGTH(telephone) >= 10), -- Fixed spelling
    CONSTRAINT valid_email EXPECT (email IS NOT NULL),
    CONSTRAINT valid_date_of_birth EXPECT (CAST(date_of_birth AS DATE) >= CAST('1920-01-01' AS DATE))
)
COMMENT 'cleaned customer data'
TBLPROPERTIES('quality' = 'silver')
AS
SELECT customer_id,
       customer_name,
       CAST(date_of_birth AS DATE) AS date_of_birth,
       telephone,
       email,
       CAST(created_date AS DATE) AS created_date
FROM STREAM(LIVE.bronze_customers);

In [0]:
-- to silver with SCD first we need to have a table 

create or refresh streaming table silver_customer
comment ' cleaned customer data'
tblproperties ('quality' = 'silver')

In [0]:
Apply changes into LIVE.silver_customer
from STREAM(LIVE.silver_customer_clean)
keys (customer_id)
sequence by created_date
stored as SCD TYPE 1; -- optional. type-1 is a default value